# Question answering — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x, axis=-1):
    x=np.asarray(x, dtype=float); x=x-x.max(axis=axis, keepdims=True); e=np.exp(x); return e/e.sum(axis=axis, keepdims=True)
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x, dtype=float)))
def normalize(v):
    v=np.asarray(v, dtype=float); n=np.linalg.norm(v); return v/(n if n else 1)
def edit_distance(a,b):
    D=np.zeros((len(a)+1,len(b)+1), dtype=int); D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1, D[i,j-1]+1, D[i-1,j-1]+(a[i-1]!=b[j-1]))
    return D
print('setup ok')

## Select or generate an answer conditioned on a question and context
The examples compute span start/end scores, no-answer thresholding, retrieval similarity, extractive span consistency and generative confidence.

In [ ]:
toks='Paris is in France'.split(); start=softmax([0.1,0.2,0.1,2.0]); end=softmax([0.1,0.1,0.2,2.5])
plt.figure(figsize=(6,3)); plt.plot(start,'-o',label='start'); plt.plot(end,'-s',label='end'); plt.xticks(range(4),toks); plt.legend(); plt.title('Extractive QA predicts start and end')
assert int(np.argmax(start))==3 and int(np.argmax(end))==3

In [ ]:
S=np.outer(start,end); S=np.triu(S)
plt.figure(figsize=(4,3)); plt.imshow(S,cmap='Blues'); plt.title('Span score requires end after start')
assert np.argmax(S)==15

In [ ]:
no_answer=0.62; best_span=0.55
plt.figure(figsize=(4,3)); plt.bar(['no answer','best span'],[no_answer,best_span]); plt.title('Threshold can abstain')
assert no_answer>best_span

In [ ]:
q=np.array([1.,0.]); docs=np.array([[.9,.1],[0,1],[.7,.3]]); sims=docs@q/(np.linalg.norm(docs,axis=1)*np.linalg.norm(q))
plt.figure(figsize=(5,3)); plt.bar(['d0','d1','d2'],sims); plt.title('Retriever selects relevant context')
assert int(np.argmax(sims))==0

In [ ]:
logits=np.array([3,1,0.5]); p=softmax(logits)
plt.figure(figsize=(4,3)); plt.bar(['Paris','London','Rome'],p); plt.title('Generative QA confidence over answers')
assert p[0]>0.8